#**💻Extreme Event Analysis: Frequency, Mean, and Max Intensity**

> **Updated** 11-Nov-2025 <br/>
> **Team** ART(AI-based prediction Research and Technology)/APCC(APEC Climate Center)<br/>
> **Contact** Miae Kim (miaekim@apcc21.org)
<br/>

In this notebook, you analyzes weekly averaged or accumulated climate extreme events from ERA5 over East Asia. You calculates climatological extreme precipitation statistics (frequency, mean intensity, max intensity) and visualizes them spatially.

## **Import libraries**

In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs

## **Set paths**

In [ ]:
PATH_data = "/Path-to-data/"
PATH_clim = "/Path-to-climatology/"
PATH_save_stat = "/Path-to-save-stats/"
PATH_save_fig = "/Path-to-save-figs/"

## **Analysis Settings**

Percentile threshold for extremes, Climatology reference period


In [ ]:
boundary = 0.90
ref_syear, ref_eyear = 1991, 2020

## **Load ERA5 Data and Compute 7-Day Rolling Averages / Sums**

- Precipitation : apply a rolling window sum over 7 days

In [ ]:
ds = xr.open_dataset(os.path.join(PATH_data, "ERA5_daily_TP_stacked_1940_2024.nc"))
ds = ds.sortby('lat', ascending=False)

window = 7
rolling0 = ds.tp.rolling(time=window, center=False).sum()
rollings = rolling0.shift({'time': -6})  # shift to align rolling window end with date
rollings = rollings.assign_coords(doy=rollings.time.dt.dayofyear)
rollings = rollings.dropna(dim='time', how='all')

rollings *= 1000  # convert m to mm

## **Load Climatological Threshold for Extreme Event**

Load precomputed threshold values (i.e., 90th percentile).


In [ ]:
threshold = xr.open_dataarray(
    os.path.join(PATH_clim, f"ERA5_climatology_TP_{int(boundary*100)}th_{ref_syear}_{ref_eyear}.nc")
)

## **Calculate Extreme Event Frequency per Year**

Identify days exceeding the threshold and count extreme events by year.


In [ ]:
extreme_mask = rollings.groupby('doy') >= threshold
extreme_counts = extreme_mask.groupby('time.year').sum(dim='time')
extreme_counts_avg = extreme_counts.mean(dim='year')

## **Plot Average Annual Frequency of Extreme Events**

Focus on East Asia extent.


In [ ]:
lon_min, lon_max = 114, 141
lat_min, lat_max = 21, 48

fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection=ccrs.PlateCarree())

im = extreme_counts_avg.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='YlGnBu',
    vmin=20, vmax=55,
    cbar_kwargs={"shrink": 0.7, 'pad': 0.02, 'location': 'bottom'}
)

ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.coastlines()
ax.set_title("Frequency [times/yr]", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## **Calculate Mean Intensity of Extreme Events (mm/yr)**

Calculate mean precipitation intensity on days flagged as extreme.


In [ ]:
extreme = rollings.where(extreme_mask)
extreme_avg = extreme.groupby('time.year').mean(dim='time')
extreme_avg_avg = extreme_avg.mean(dim='year')

## **Plot Average Annual Mean Intensity of Extreme Events**


In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection=ccrs.PlateCarree())

im = extreme_avg_avg.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='YlGnBu',
    vmax=200,
    cbar_kwargs={"shrink": 0.7, 'pad': 0.02, 'location': 'bottom'}
)

ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.coastlines()
ax.set_title("Mean Intensity [mm/yr]", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## **Calculate and Plot Mean Maximum Intensity of Extreme Events (mm/yr)**


In [ ]:
extreme_max = extreme.groupby('time.year').max(dim='time')
extreme_max_avg = extreme_max.mean(dim='year')

fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection=ccrs.PlateCarree())

im = extreme_max_avg.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='YlGnBu',
    vmax=400,
    cbar_kwargs={"shrink": 0.7, 'pad': 0.02, 'location': 'bottom'}
)

ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.coastlines()
ax.set_title("Max Intensity [mm/yr]", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## **Combined Visualization & Save Figure**

In [ ]:
plot_info = [
    (extreme_counts_avg, "Frequency [times/yr]", 20, 55, "YlGnBu"),
    (extreme_avg_avg, "Mean Intensity [mm/yr]", None, 200, "YlGnBu"),
    (extreme_max_avg, "Max Intensity [mm/yr]", None, 400, "YlGnBu"),
]

fig, axes = plt.subplots(
    nrows=1, ncols=3,
    figsize=(10, 4),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

for ax, (data, title, vmin, vmax, cmap) in zip(axes, plot_info):
    im = data.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False
    )
    ax.coastlines()
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=15, fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, orientation="horizontal", pad=0.02)

plt.tight_layout()
# plt.savefig(os.path.join(PATH_save_fig, f"TP_extreme_maps_{ref_syear}_{ref_eyear}_p{int(boundary*100)}.png"), dpi=100, bbox_inches='tight')
plt.show()

## **Save Combined Dataset with Metadata (NetCDF)**

Prepare dataset with frequency, mean intensity, and max intensity including metadata.


In [ ]:
ds_extremes = xr.Dataset(
    {
        "frequency": extreme_counts_avg,
        "mean_intensity": extreme_avg_avg,
        "mean_max_intensity": extreme_max_avg,
    }
)

# Add metadata attributes
ds_extremes.frequency.attrs["units"] = "times per year"
ds_extremes.mean_intensity.attrs["units"] = "mm per year"
ds_extremes.mean_max_intensity.attrs["units"] = "mm per year"

ds_extremes.attrs["description"] = (
    "Climatological statistics of 7-day extreme precipitation: "
    "frequency, mean intensity, and mean of annual maxima"
)

ds_extremes.attrs["reference_period"] = f"{ref_syear}-{ref_eyear}"

# === Save
# out_path = os.path.join(PATH_save_stat, f"TP_extreme_stats_{ref_syear}_{ref_eyear}_p{int(boundary*100)}.nc")
# ds_extremes.to_netcdf(out_path)
# print(f"Saved combined dataset to {out_path}")